In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns


In [ ]:

foods = pd.read_csv(
    "FoodData_Central_branded_food_csv_2026-04-30/food.csv",
    low_memory= False
)

nutrients = pd.read_csv(
    "FoodData_Central_branded_food_csv_2026-04-30/food_nutrient.csv",
    low_memory= False
)


foods.columns = foods.columns.str.strip()

protein = nutrients[nutrients['nutrient_id'] == 1003][['fdc_id', 'amount']]
calories = nutrients[nutrients['nutrient_id'] == 1008][['fdc_id', 'amount']]

nutrition = pd.merge(protein, calories, on="fdc_id")

df = pd.merge(
    foods[['fdc_id', 'description']],
    nutrition,
    on="fdc_id"
)


df = df.dropna()

df['amount_x'] = pd.to_numeric(df['amount_x'], errors='coerce')
df['amount_y'] = pd.to_numeric(df['amount_y'], errors='coerce')

df = df.dropna()

df = df[df['amount_x'] > 0]
df = df[df['amount_y'] > 0]
df = df[df['amount_x'] < 100]
df = df[df['amount_y'] < 2000]

df = df.reset_index(drop = True)
df = df.drop_duplicates(subset = 'description', keep = 'first')

df['protein_efficiency'] = df['amount_x'] / df['amount_y']

In [ ]:

top10 = df.sort_values(by= 'protein_efficiency', ascending = False).head(10)

top10 = top10.reset_index(drop =True)
top10.index = top10.index + 1

defined = top10.rename( columns = {
    'amount_x': 'protein_g',
    'amount_y': 'calories_kcal'
})

print("Top 10 Branded High Protein and Low Calorie Foods:")
print(defined[['description', 'protein_g', 'calories_kcal', 'protein_efficiency']])


In [ ]:
worst10 = df.sort_values(by = 'protein_efficiency').head(10)

worst10 = worst10.reset_index(drop= True)
worst10.index = worst10.index + 1

print("Top 10 Worst Protien Efficiency Foods:")
print(worst10[['description', 'protein_efficiency']])

In [ ]:
highest_protein = defined.sort_values(by = 'protein_g', ascending = False).head(10)


highest_protein = highest_protein.reset_index (drop = True)
highest_protein.index = highest_protein.index + 1

print("\nHighest Protein Foods:\n")
print(highest_protein[['description', 'protein_g', 'calories_kcal']])

In [ ]:
connect = defined[['protein_g', 'calories_kcal', 'protein_efficiency']].corr()

print ("\nHow these catagories Correlate:\n")
print(connect)

In [ ]:
defined = df.copy()

def categorize(text):
    
    text = str(text).lower()

    if any(word in text for word in ["chicken", "beef", "turkey", "pork", "steak", "jerky"]):

        return "Meat"
    elif any(word in text for word in  ["milk", "cheese", "yogurt", "cream"]):

        return "Dairy"
    elif any(word in text for word in ["tofu", "soy", "bean", "lentil"]):

        return "Plant-Based"
    elif any(word in text for word in ["peanut", "almond", "cashew"]):

        return "Nut-Based"
    elif any(word in text for word in ["bar", "snack", "chip", "cracker"]):

        return "Snack"
    elif any(word in text for word in  ["drink", "juice", "beverage", "shake"]):

        return "Beverage"
    
    elif "protein" in text:
        return "Protein Product"
    
    elif any(word in text for word in ["pizza", "pasta", "meal", "chili", "rice"]):

        return "Prepared Food"
    else:
        return "Other"



defined['category'] =  defined['description'].apply(categorize)

category =  defined.groupby('category')['protein_efficiency'].mean().sort_values(ascending=False)



print("\nAverage Protein Efficiency by Category:\n")
print(category)

In [ ]:
plt.figure(figsize=(10, 6))

ax = sns.barplot(x=category.values, y=category.index, hue = category.index, palette="magma", legend=False)

plt.title("Protein Efficiency: Which Categories Win?", fontsize = 14, fontweight = 'bold')
plt.xlabel("Average Protein Grams per Calorie")
plt.ylabel("Food Category")


for i in ax.containers:
    ax.bar_label(i, fmt='%.3f', padding= 3)

plt.tight_layout()
plt.show()

In [ ]:
if 'category' not in df.columns:
    df['category'] = df['description'].apply(categorize)



x_name = 'calories' if 'calories' in df.columns else 'amount_y'
y_name = 'protein' if 'protein' in df.columns else 'amount_x'


plt.figure(figsize=(12, 7))


sns.scatterplot(
    data=df, 
    x=x_name, 
    y=y_name, 
    hue='category', 
    alpha=0.5, 
    s=20,
    palette='viridis' 
)

plt.title("Protein vs. Calories: High Density Foods", fontsize=15, pad=15)
plt.xlabel("Calories (kcal)", fontsize=12)
plt.ylabel("Protein (g)", fontsize=12)

plt.legend(title='Food Category', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))


plt.hist(df['protein_efficiency'], bins=100, range=(0, 1), color='skyblue', edgecolor='black', alpha=0.7)

plt.axvline(0.25, color='green', linestyle=':', linewidth=2, label='Max Theoretical (Pure Protein)')


plt.title("Protein Efficiency Distribution (0-1 Scale)", fontsize=14)
plt.xlabel("Efficiency Ratio (0.25 = Pure Protein)")
plt.ylabel("Number of Food Items")

plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

colors = sns.color_palette("magma", len(category))
ax = category.plot(kind='bar', color=colors, edgecolor='black', alpha=0.8)

for i, v in enumerate(category):
    ax.text(i, v + 0.002, f'{v:.3f}', ha='center', fontweight='bold')

plt.title("Which Categories are the Most Protein-Efficient?", fontsize=15, pad=20)
plt.ylabel("Protein Efficiency (g per kcal)", fontsize=12)
plt.xlabel("Food Category", fontsize=12)

plt.xticks(rotation=45, ha='right')

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout() 
plt.show()

In [ ]:
x_name = 'calories' if 'calories' in df.columns else 'amount_y'
y_name = 'protein' if 'protein' in df.columns else 'amount_x'

x = df[x_name].values
y = df[y_name].values


m, b = np.polyfit(x, y, 1)

print("LINEAR MODEL RESULTS")
print(f"Slope (Protein per Calorie): {m:.4f}")
print(f"Intercept (Base Protein):    {b:.4f}")
print(f"Interpretation: For every 100 calories, you average {m*100:.1f}g of protein.")


plt.figure(figsize=(10, 6))
plt.scatter(x, y, alpha=0.2, label='Actual Data', s=10)


plt.plot(x, m*x + b, color='red', linewidth=2, label=f'Trendline (y={m:.3f}x + {b:.2f})')

plt.title("Protein vs. Calories with Linear Regression", fontsize=14)
plt.xlabel("Calories")
plt.ylabel("Protein (g)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
y_predict = m * x + b

plot_total = np.sum((y - np.mean(y))**2)  
plot_res = np.sum((y - y_predict)**2)

r2 = 1 - (plot_res / plot_total)

print("MODEL ACCURACY SUMMARY")


print(f"R^2 Score: {r2:.4f}")

if r2 > 0.7:
    
    print("Strong correlation. Calories are a good predictor of protein content.")
elif r2 > 0.4:

    print("Moderate correlation. Other factors (like food category) matter a lot.")
else:

    print("Weak correlation. Protein content varies wildly regardless of calorie count.")


leftout = y - y_predict
print(f"\nMax Over-prediction: {np.min(leftout):.2f}g protein")
print(f"Max Under-prediction: {np.max(leftout):.2f}g protein")


In [ ]:
m = np.mean(leftout**2)
rm = np.sqrt(m)

print(f"Mean Error: {m:.4f}")
print(f"Typical Prediction Error: {rm:.2f}g")
print(f"On average, my linear model is off by about {rm:.2f} grams of protein.")

In [ ]:
auditing = {

    "Initial Raw Rows": len(foods),
    "Rows after Null Removal": len(df) + (len(foods) - len(df)), 
    "Duplicates Removed": df.description.duplicated().sum(),
    "Final Cleaned Sample Size": len(df)
}

print("\nDATA SIZE AUDITING")

for key, value in auditing.items():
    print(f"{key}: {value}")

In [ ]:
defined.to_csv('cleaned_food_data.csv', index=False)